## Aim: Retrieve the same insights from the data as Spotify Wrapped does

Namely top artists and songs, as well as beginning to use spotipy api to determine top genres

In [7]:
# relevant imports

import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import requests

df = pd.read_csv('Spotify Extended Streaming History/cleaned-history.csv')

In [55]:
import os
import dotenv

dotenv.load_dotenv()
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

In [56]:
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=client_id,
    client_secret=client_secret,
    redirect_uri="http://127.0.0.1:8888/callback",
    scope="user-read-private" # basic access scope
))

In [9]:
df['ts'] = pd.to_datetime(df['ts'], utc=True)
def historySplit(df, year):
    return df[df['ts'].dt.year == year]

# Part 1: Top Artists by Time Played 

In [57]:
# get all unique artists and assign them as column in a new df
allArtists = df['master_metadata_album_artist_name'].unique()
topArtists = pd.DataFrame({'Artists': allArtists})

In [58]:
# sum each artist's total ms_played and assign it to a new column in the topArtists df
topArtists['time_played'] = topArtists['Artists'].apply(lambda x: df[df['master_metadata_album_artist_name'] == x]['ms_played'].sum())

In [59]:
# rank from most to least time played and reset index
topArtists.sort_values(by='time_played', ascending=False, inplace=True)
topArtists.reset_index(drop=True, inplace=True)

In [60]:
topArtists['time_played'] = topArtists['time_played'].apply(lambda x: x / 60000)  # convert ms to minutes
topArtists['time_played'] = topArtists['time_played'].apply(lambda x: round(x, 2))  # round to 2 decimal places

In [61]:
print("Top 10 Artists by Time Played (in minutes):")
print(topArtists.head(10))

Top 10 Artists by Time Played (in minutes):
                            Artists  time_played
0                          Gorillaz     11466.21
1                         Radiohead     11061.90
2                     Elliott Smith      9644.36
3  King Gizzard & The Lizard Wizard      9576.21
4                       The Beatles      8115.76
5                        Aphex Twin      7846.62
6                 Car Seat Headrest      7458.82
7                 Father John Misty      7079.84
8                       Death Grips      7038.68
9                    Kendrick Lamar      6678.78


## Part 2: Top Songs by Time Listened

In [21]:
allSongs = df['master_metadata_track_name'].unique()
topSongs = pd.DataFrame({'Songs': allSongs})

In [22]:
topSongs['time_played'] = topSongs['Songs'].apply(lambda x: df[df['master_metadata_track_name'] == x]['ms_played'].sum())

In [23]:
topSongs.sort_values(by='time_played', ascending=False, inplace=True)
topSongs.reset_index(drop=True, inplace=True)

In [24]:
topSongs['time_played'] = topSongs['time_played'].apply(lambda x: x/ 60000)
topSongs['time_played'] = topSongs['time_played'].apply(lambda x: round(x,2))

In [25]:
print("Top 20 songs by time played (in minutes):")
print(topSongs.head(20))

Top 20 songs by time played (in minutes):
                                    Songs  time_played
0   The Place Where He Inserted the Blade      1339.42
1                          All My Friends      1161.14
2                           Blackest Bile      1060.30
3                                Souk Eye       978.32
4                    God Is in the Rhythm       946.21
5                                DAYDREAM       939.80
6                     Beach Life-In-Death       919.76
7                               Risingson       919.12
8                               sometimes       909.96
9                                      #3       886.77
10                              styrofoam       879.19
11                              Fireworks       862.67
12       Echoes - 2011 Remastered Version       846.58
13          In the Aeroplane Over the Sea       843.64
14                     Dance Yrself Clean       842.79
15                           Windowlicker       838.58
16                     

Using the same approach to titles as with artists does not work as effectively because there are songs that share a title (i.e. Kids by Current Joys and Kids by MGMT) that have combined into one entry. I will change the approach to consider each song individually

## Part 2a: Top Tracks, Properly Filtered by Artist

In [ ]:
# get all unique track and artist pairings from dataframe
artists_tracks = df[['master_metadata_album_artist_name', 'master_metadata_track_name']].drop_duplicates()

In [43]:
# sum each track's total ms_played and assign it to a new column in the artists_tracks df
artists_tracks['time_played'] = artists_tracks.apply(lambda row: df[(df['master_metadata_album_artist_name'] == row['master_metadata_album_artist_name']) & (df['master_metadata_track_name'] == row['master_metadata_track_name'])]['ms_played'].sum(), axis=1)

In [46]:
artists_tracks.sort_values(by='time_played', ascending=False, inplace=True)
artists_tracks.reset_index(drop=True, inplace=True)

In [47]:
artists_tracks['time_played'] = artists_tracks['time_played'].apply(lambda x: x / 60000)  # convert ms to minutes
artists_tracks['time_played'] = artists_tracks['time_played'].apply(lambda x: round(x, 2))  # round to 2 decimal places

In [51]:
print("Top 20 Songs by Time Played (in minutes):")
artists_tracks.head(20)

Top 20 Songs by Time Played (in minutes):


,master_metadata_album_artist_name,master_metadata_track_name,time_played
0,"Black Country, New Road",The Place Where He Inserted the Blade,1339.42
1,LCD Soundsystem,All My Friends,1161.14
2,Giles Corey,Blackest Bile,1060.30
3,Gorillaz,Souk Eye,978.32
4,King Gizzard & The Lizard Wizard,God Is in the Rhythm,946.21
5,Fishmans,DAYDREAM,939.80
6,Car Seat Headrest,Beach Life-In-Death,919.76
7,Massive Attack,Risingson,919.12
8,my bloody valentine,sometimes,909.96
9,Aphex Twin,#3,886.77


## Part 3: Top Genres

In [62]:
genreArtists = topArtists.head(150) # limit to top 150 artists so as to not exceed API limit  

In [ ]:
# get genre for each artist using Last.fm API
dotenv.load_dotenv()
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")

artist_genre_lookup = {}

url = "http://ws.audioscrobbler.com/2.0/"

# Iterate through each artist and fetch their top tags (genres) from Last.fm
for index, artist_name in enumerate(genreArtists['Artists']):
    try:
        # Prepare the payload for the Last.fm API request
        payload = {
            'method': 'artist.getTopTags',
            'artist': artist_name,
            'api_key': LASTFM_API_KEY,
            'format': 'json'
        }
        
        # Make the API request to Last.fm
        response = requests.get(url, params=payload)
        data = response.json()
        
        # Extract the top tags (genres) from the response
        toptags_container = data.get('toptags', {})
        raw_tags_list = toptags_container.get('tag', [])
        
        # Extract the top 3 genres (if available) and convert them to lowercase
        genres = [tag['name'].lower() for tag in raw_tags_list[:3]]
        
        
        if not genres:
            genres = ['unknown']
            
        artist_genre_lookup[artist_name] = genres
    
    # Handle specific exceptions
    except Exception as e:
        print(f"Skipping {artist_name} due to an unexpected error: {e}")
        continue

In [ ]:
# Map the genres to the genreArtists DataFrame
genreArtists['genres'] = genreArtists['Artists'].map(artist_genre_lookup)
# Fill any missing genres with 'unknown'
genreArtists['genres'] = genreArtists['genres'].fillna({i: ['unknown'] for i in genreArtists.index})

In [ ]:
# Explode the genres list into separate rows for each genre
genreArtists = genreArtists.explode('genres', ignore_index=True)

In [ ]:
# Sum the time played for each genre and sort in descending order
genre_sum = genreArtists.groupby('genres')['time_played'].sum().reset_index()
genre_sum.sort_values(by='time_played', ascending=False, inplace=True)
genre_sum.reset_index(drop=True, inplace=True)

In [69]:
print("Top 20 Genres by Time Played (in minutes):")
print(genre_sum.head(20))

Top 20 Genres by Time Played (in minutes):
               genres  time_played
0             hip-hop     82075.99
1               indie     75318.46
2          electronic     70196.39
3                rock     63447.21
4         alternative     45251.10
5                 rap     44406.51
6        experimental     38797.13
7                folk     36143.27
8           post-punk     30371.53
9   singer-songwriter     29451.09
10         indie rock     28776.76
11            ambient     25566.31
12       classic rock     24517.70
13              lo-fi     22002.23
14                pop     21492.52
15   alternative rock     21372.54
16            british     19544.01
17          indie pop     17507.25
18           shoegaze     17413.73
19            hip hop     16903.11


## Part 4: Track and Artist Functions

In [1]:
def topTracks(df):
    artists_tracks = df[['master_metadata_album_artist_name', 'master_metadata_track_name']].drop_duplicates()
    artists_tracks['time_played'] = artists_tracks.apply(lambda row: df[(df['master_metadata_album_artist_name'] == row['master_metadata_album_artist_name']) & (df['master_metadata_track_name'] == row['master_metadata_track_name'])]['ms_played'].sum(), axis=1)
    artists_tracks.sort_values(by='time_played', ascending=False, inplace=True)
    artists_tracks.reset_index(drop=True, inplace=True)
    artists_tracks['time_played'] = artists_tracks['time_played'].apply(lambda x: x / 60000)  # convert ms to minutes
    artists_tracks['time_played'] = artists_tracks['time_played'].apply(lambda x: round(x, 2))  # round to 2 decimal places
    return artists_tracks

In [10]:
history2023 = historySplit(df, 2023)

In [11]:
topTracks2023 = topTracks(history2023)

In [12]:
print("Top 20 Tracks of 2023 by Time Played (in minutes):")
print(topTracks2023.head(20))

Top 20 Tracks of 2023 by Time Played (in minutes):
   master_metadata_album_artist_name             master_metadata_track_name  \
0                    LCD Soundsystem                         All My Friends   
1   King Gizzard & The Lizard Wizard                   God Is in the Rhythm   
2                           Fishmans                               DAYDREAM   
3        Godspeed You! Black Emperor                    The Dead Flag Blues   
4                      Elliott Smith                        The Biggest Lie   
5   King Gizzard & The Lizard Wizard                              The River   
6                      Talking Heads  This Must Be the Place (Naive Melody)   
7                    LCD Soundsystem                     Dance Yrself Clean   
8   King Gizzard & The Lizard Wizard                           Head On/Pill   
9                       Machine Girl                 MRK90 MIX VOL 1 SIDE A   
10                            Pixies                                Debaser   
1

In [13]:
def topArtists(df):
    allArtists = df['master_metadata_album_artist_name'].unique()
    topArtists = pd.DataFrame({'Artists': allArtists})
    topArtists['time_played'] = topArtists['Artists'].apply(lambda x: df[df['master_metadata_album_artist_name'] == x]['ms_played'].sum())
    topArtists.sort_values(by='time_played', ascending=False, inplace=True)
    topArtists.reset_index(drop=True, inplace=True)
    topArtists['time_played'] = topArtists['time_played'].apply(lambda x: x / 60000)
    topArtists['time_played'] = topArtists['time_played'].apply(lambda x: round(x, 2))
    return topArtists

In [14]:
topArtists2023 = topArtists(history2023)

In [16]:
print("Top 20 Artists of 2023 by Time Played (in minutes):")
print(topArtists2023.head(20))

Top 20 Artists of 2023 by Time Played (in minutes):
                             Artists  time_played
0   King Gizzard & The Lizard Wizard      5565.70
1                      Elliott Smith      4010.59
2                    LCD Soundsystem      3061.97
3                         Aphex Twin      2256.72
4            Queens of the Stone Age      1862.02
5                  Father John Misty      1853.96
6                      Talking Heads      1769.58
7                        Death Grips      1668.75
8                      Lil Ugly Mane      1451.04
9                          Jockstrap      1358.89
10                      Machine Girl      1285.09
11                         Radiohead      1266.81
12                            Pixies      1194.83
13           Black Country, New Road      1172.95
14                 Car Seat Headrest      1106.07
15                         JPEGMAFIA      1088.50
16                          slowthai      1024.58
17                   Nine Inch Nails      1006.9